In [36]:

import requests
import re
import pandas as pd
import streamlit as st
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [37]:
#reading our api key file
with open('api.txt') as file:
    api_key = file.read().strip()
query = "\"Legumes and Legume Products\""

#creating a list from 5 pages of data from our API call
legus = []
for page in range(1,6):
    url = f'https://api.nal.usda.gov/fdc/v1/foods/list?api_key={api_key}&query={query}&pageNumber={page}'
    response = requests.get(url)
    legu_rows = response.json()
    legus.extend(legu_rows)

legusdf = pd.DataFrame(legus)

In [38]:
legusdf = legusdf[legusdf["dataType"] == "Foundation"]

legusdf = legusdf.reset_index(drop = True)

# legusdf.head(50)

In [39]:
def extract_nutrient(nutrient_list, target):
    if nutrient_list is None:
        return None
    if len(nutrient_list) == 0:
        return None
    for nutrient in nutrient_list:
        if nutrient.get('name') == target:
            return nutrient.get("amount")
    return None

In [40]:
legusdf["Water (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Water"))
legusdf["Calories (kcal)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Energy (Atwater General Factors)"))
legusdf["Nitrogen (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Nitrogen"))
legusdf["Protein (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Protein"))
legusdf["Fat (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Total lipid (fat)"))
legusdf["Ash (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Ash"))
legusdf["Carbs (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Carbohydrate, by difference"))
legusdf["Starch (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Starch"))
legusdf["Resistant starch (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Resistant starch"))
legusdf["Calcium (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Calcium, Ca"))
legusdf["Iron (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Iron, Fe"))
legusdf["Magnesium (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Magnesium, Mg"))
legusdf["Phosphorus (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Phosphorus, P"))
legusdf["Potassium (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Potassium, K"))
legusdf["Sodium (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Sodium, Na"))
legusdf["Zinc (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Zinc, Zn"))
legusdf["Copper (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Copper, Cu"))
legusdf["Manganese (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Manganese, Mn"))


In [41]:
legusdf = legusdf.fillna(0.0)

legusdf = legusdf.round(2)

legusdf['Category'] = legusdf["description"].str.extract(r'^(.*?),')
legusdf['Type'] = legusdf["description"].str.extract(r',\s*(.*)')


legusdf = legusdf[[
    "fdcId",
    "Category",
    "Type",
    "description",
    "Water (g)",
    "dataType",
    "publicationDate",
    "ndbNumber",
    "foodNutrients",
    "Calories (kcal)",
    "Nitrogen (g)",
    "Protein (g)",
    "Fat (g)",
    "Carbs (g)",
    "Ash (g)",
    "Starch (g)",
    "Resistant starch (g)",
    "Iron (mg)",
    "Magnesium (mg)",
    "Phosphorus (mg)",
    "Potassium (mg)",
    "Sodium (mg)",
    "Zinc (mg)",
    "Copper (mg)",
    "Manganese (mg)"
]]

legusdf = legusdf.drop(columns = ["fdcId", "description", "dataType", "ndbNumber", "foodNutrients", "publicationDate"])
legusdf


,Category,Type,Water (g),Calories (kcal),Nitrogen (g),Protein (g),Fat (g),Carbs (g),Ash (g),Starch (g),Resistant starch (g),Iron (mg),Magnesium (mg),Phosphorus (mg),Potassium (mg),Sodium (mg),Zinc (mg),Copper (mg),Manganese (mg)
0,Beans,"black, canned, sodium added, drained and rinsed",70.80,118.0,1.11,6.91,1.27,19.8,1.21,11.80,2.0,1.69,32.5,91.0,253.0,218.00,0.74,0.34,0.33
1,Beans,"cannellini, canned, sodium added, drained and ...",71.50,115.0,1.19,7.41,1.17,18.8,1.07,11.30,2.0,1.40,29.3,94.7,203.0,164.00,0.52,0.22,0.37
2,Beans,"cannellini, dry",12.30,345.0,3.45,21.60,2.20,59.8,4.14,29.90,33.0,6.67,154.0,412.0,1420.0,0.00,2.72,0.74,1.78
3,Beans,"Dry, Black (0% moisture)",0.00,0.0,0.00,24.40,1.45,0.0,0.00,36.60,0.0,5.34,180.0,522.0,1540.0,0.00,3.37,1.12,2.08
4,Beans,"Dry, Brown (0% moisture)",0.00,0.0,0.00,25.60,1.12,0.0,0.00,37.20,0.0,4.70,158.0,543.0,1580.0,0.00,3.71,1.03,1.87
5,Beans,"Dry, Carioca (0% moisture)",0.00,0.0,0.00,25.20,1.44,0.0,0.00,36.00,0.0,5.87,179.0,535.0,1390.0,0.00,3.36,1.04,1.93
6,Beans,"Dry, Cranberry (0% moisture)",0.00,0.0,0.00,24.40,1.23,0.0,0.00,36.30,0.0,5.26,166.0,487.0,1340.0,0.00,3.03,0.76,1.78
7,Beans,"Dry, Dark Red Kidney (0% moisture)",0.00,0.0,0.00,25.90,1.31,0.0,0.00,36.70,0.0,6.58,164.0,612.0,1490.0,0.00,3.29,0.86,1.67
8,Beans,"Dry, Flor de Mayo (0% moisture)",0.00,0.0,0.00,23.30,0.86,0.0,0.00,40.00,0.0,4.47,182.0,439.0,1490.0,0.00,3.42,0.86,1.70
9,Beans,"Dry, Great Northern (0% moisture)",0.00,0.0,0.00,24.70,1.24,0.0,0.00,37.90,0.0,5.48,176.0,519.0,1520.0,0.00,3.45,1.08,1.90


In [42]:
# Corelation Heat Map

numeric = legusdf.select_dtypes(include="number")

corr = numeric.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr, 
    mask=mask, 
    annot=True, 
    cmap="coolwarm", 
    fmt=".2f", 
    linewidths=1.0
)

plt.title("Correlation Heatmap (Bottom Triangle Only)")
plt.show()



/var/folders/62/1s75npnx2q348wkv6mmzbvj00000gn/T/ipykernel_60848/3767666418.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [43]:
# Box Plot by nutrient for beans category
beans = legusdf[legusdf["Category"] == "Beans"]

nutrients = ["Protein (g)", "Fat (g)", "Carbs (g)", "Calories (kcal)"]

plt.figure(figsize=(12, 8))

for i, nutrient in enumerate(nutrients, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(y=beans[nutrient])
    plt.title(f"{nutrient} Distribution (Beans Only)")
    plt.ylabel(nutrient)
    plt.xlabel("Beans")

plt.tight_layout()
plt.show()


/var/folders/62/1s75npnx2q348wkv6mmzbvj00000gn/T/ipykernel_60848/2677653678.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [44]:
beans = legusdf[legusdf["Category"] == "Beans"]

filtered_beans = beans[(beans["Protein (g)"] < 150) &
                       (beans["Fat (g)"] < 150) &
                       (beans["Carbs (g)"] < 150) &
                       (beans["Calories (kcal)"] < 150)]

# Select nutrients
nutrients = ["Protein (g)", "Fat (g)", "Carbs (g)", "Calories (kcal)"]

# Melt the dataframe from wide → long format
beans_melted = filtered_beans.melt(value_vars=nutrients, var_name="Nutrient", value_name="Value")

plt.figure(figsize=(10, 6))
sns.boxplot(data=beans_melted, x="Nutrient", y="Value")
plt.title("Nutrient Distributions for Beans")
plt.xlabel("Nutrient")
plt.ylabel("Value (per 100g)")

plt.show()

/var/folders/62/1s75npnx2q348wkv6mmzbvj00000gn/T/ipykernel_60848/971515004.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [45]:
# Average Mineral content (Bar Plot)
minerals = ["Nitrogen (g)",
    "Protein (g)",
    "Fat (g)",
    "Ash (g)",
    "Carbs (g)",
    "Starch (g)",
    "Resistant starch (g)",
    "Iron (mg)",
    "Magnesium (mg)",
    "Phosphorus (mg)",
    "Potassium (mg)",
    "Sodium (mg)",
    "Zinc (mg)",
    "Copper (mg)",
    "Manganese (mg)"]

# Compute means
mineral_means = legusdf[minerals].mean().reset_index()
mineral_means.columns = ["Mineral", "Average_Value"]

# Plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=mineral_means, x="Mineral", y="Average_Value")

plt.title("Average Mineral Content Across All Legumes")
plt.xlabel("Mineral")
plt.xticks(rotation=45, fontsize=10)
plt.ylabel("Average Amount (per 100g sample)")

for p in ax.patches:
    value = p.get_height()
    x = p.get_x() + p.get_width() / 2
    y = value
    ax.text(x, y, f"{value:.1f}", ha="center", va="bottom", fontsize=9)
    
plt.tight_layout()
plt.show()

/var/folders/62/1s75npnx2q348wkv6mmzbvj00000gn/T/ipykernel_60848/2628343372.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [46]:
# Make a csv file
legusdf.to_csv("legus_cleaned.csv", index=False)